In [ ]:
import json
import logging

from openeo.rest.udp import build_process_dict
from utils import udp_params, urls, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = udp_params.SPATIAL_EXTENT

canopy_cover_threshold = udp_params.CANOPY_COVER_THRESHOLD
natural_forest_threshold = udp_params.NATURAL_FOREST_THRESHOLD
min_connected_area = udp_params.MIN_CONNECTED_AREA

s1_orbit_state = udp_params.S1_ORBIT_STATE
s1_relative_orbit = udp_params.S1_RELATIVE_ORBIT
speckle_filter_radius = udp_params.SPECKLE_FILTER_RADIUS
speckle_filter_cv_noise = udp_params.SPECKLE_FILTER_CV_NOISE
speckle_filter_temporal_window = udp_params.SPECKLE_FILTER_TEMPORAL_WINDOW

spatial_resolution = udp_params.SPATIAL_RESOLUTION

logistic_window_size = udp_params.LOGISTIC_WINDOW_SIZE
logistic_steepness_parameter = udp_params.LOGISTIC_STEEPNESS_PARAMETER

temporal_variability_threshold = udp_params.TEMPORAL_VARIABILITY_THRESHOLD
flattening_threshold = udp_params.FLATTENING_THRESHOLD
logistic_sse_threshold = udp_params.LOGISTIC_SSE_THRESHOLD

cropland_probability_threshold = udp_params.CROPLAND_PROBABILITY_THRESHOLD

In [ ]:
parameters = [
    spatial_extent,
    canopy_cover_threshold,
    natural_forest_threshold,
    min_connected_area,
    s1_orbit_state,
    s1_relative_orbit,
    speckle_filter_radius,
    speckle_filter_cv_noise,
    speckle_filter_temporal_window,
    spatial_resolution,
    logistic_window_size,
    logistic_steepness_parameter,
    temporal_variability_threshold,
    flattening_threshold,
    logistic_sse_threshold,
    cropland_probability_threshold,
]

# UDP

In [ ]:
forest_baseline_datacube = connection.datacube_from_process(
    "forest_baseline",
    namespace=urls.FOREST_BASELINE_UDP,
    spatial_extent=spatial_extent,
    canopy_cover_threshold=canopy_cover_threshold,
    natural_forest_threshold=natural_forest_threshold,
    min_connected_area=min_connected_area,
)

In [ ]:
sentinel_1_datacube = connection.datacube_from_process(
    "s1_logistic_processing",
    namespace=urls.S1_PROCESSING_UDP,
    spatial_extent=spatial_extent,
    orbit_state=s1_orbit_state,
    relative_orbit=s1_relative_orbit,
    speckle_filter_radius=speckle_filter_radius,
    speckle_filter_cv_noise=speckle_filter_cv_noise,
    speckle_filter_temporal_window=speckle_filter_temporal_window,
    spatial_resolution=spatial_resolution,
    logistic_window_size=logistic_window_size,
    logistic_steepness_parameter=logistic_steepness_parameter,
)

In [ ]:
decimal_year_of_deforestation_datacube = connection.datacube_from_process(
    "deforestation",
    namespace=urls.DEFORESTATION_UDP,
    forest_baseline_datacube=forest_baseline_datacube,
    sentinel_1_datacube=sentinel_1_datacube,
    spatial_extent=spatial_extent,
    spatial_resolution=spatial_resolution,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
    logistic_sse_threshold=logistic_sse_threshold,
    min_connected_area=min_connected_area,
)

In [ ]:
kpis_vector_cube = connection.datacube_from_process(
    "KPIs",
    namespace=urls.KPIS_UDP,
    decimal_year_of_deforestation_datacube=decimal_year_of_deforestation_datacube,
    spatial_extent=spatial_extent,
    spatial_resolution=spatial_resolution,
    cropland_probability_threshold=cropland_probability_threshold,
)

# Serialise UDP

In [ ]:
summary = "End-to-end forest-loss KPIs"
description = (
    "Detect deforestation from Sentinel-1 and aggregate forest-change KPIs "
    "by Uganda ADM-4 administrative unit. "
    "1. Construct a 2020 natural-forest baseline. "
    "2. Process Sentinel-1 backscatter and fit a logistic curve per pixel. "
    "3. Detect the decimal year of deforestation. "
    "4. Sum forest-stock, annual forest-loss, and forest-loss-to-cropland "
    "areas (ha) over ADM-4 geometries. "
    "The returned vector cube contains those KPIs per administrative unit."
)

udp_spec = build_process_dict(
    kpis_vector_cube,
    process_id="KPIs",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A vector cube, with columns (bands) for each KPI",
        "schema": {"type": "object", "subtype": "vector-cube"},
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)